# Dashboard Kebocoran Gas LPG (SIC 8)

Menampilkan status 5 komponen dari ESP32 di Wokwi secara live: sensor gas MQ-2, PIR, buzzer, motor stepper, dan tombol reset.

**Cara pakai**
1. Jalankan simulasi di Wokwi sampai Serial Monitor menampilkan `[MQTT] Connecting to broker.emqx.io... CONNECTED!`.
2. Jalankan sel 1 (install), lalu sel 2. Grafik diperbarui tiap 1 detik.
3. Hentikan dengan tombol **Stop** (■) di sel 2.

Broker: `broker.emqx.io` port 1883, tanpa username/password. Topic: `tsabit/wokwi/#`.


In [ ]:
# 1. Install the MQTT client (not preinstalled in Colab)
!pip install -q paho-mqtt


In [ ]:
# 2. Subscribe to the ESP32 topics and redraw the 5 status charts every second
import paho.mqtt.client as mqtt
import matplotlib.pyplot as plt
from IPython.display import clear_output
import time

time_series = []
asap_data = []
pir_data = []
buzzer_data = []
motor_data = []
button_data = []

asap_val = 0
pir_val = 0
buzzer_val = 0
motor_val = 0
button_val = 0

def on_message(client, userdata, msg):
    global asap_val, pir_val, buzzer_val, motor_val, button_val
    topic = msg.topic
    payload = msg.payload.decode('utf-8')

    if topic == "tsabit/wokwi/status/asap":
        asap_val = 1 if payload == "ADA_ASAP" else 0
    elif topic == "tsabit/wokwi/status/pir":
        pir_val = 1 if payload == "GERAKAN_DETECTED" else 0
    elif topic == "tsabit/wokwi/status/buzzer":
        buzzer_val = 1 if payload == "BUNYI" else 0
    elif topic == "tsabit/wokwi/status/motor":
        motor_val = 1 if payload == "BERPUTAR" else 0
    elif topic == "tsabit/wokwi/status/button":
        button_val = 1 if payload == "PRESSED" else 0

client = mqtt.Client(
    mqtt.CallbackAPIVersion.VERSION1,
    client_id="Colab_Full_5in1_" + str(int(time.time()))
)

client.on_message = on_message
client.connect("broker.emqx.io", 1883, 60)
client.subscribe("tsabit/wokwi/#")
client.loop_start()

plt.ion()
start_time = time.time()

try:
    while True:
        elapsed = round(time.time() - start_time, 1)
        time_series.append(elapsed)
        asap_data.append(asap_val)
        pir_data.append(pir_val)
        buzzer_data.append(buzzer_val)
        motor_data.append(motor_val)
        button_data.append(button_val)

        if len(time_series) > 20:
            time_series.pop(0)
            asap_data.pop(0)
            pir_data.pop(0)
            buzzer_data.pop(0)
            motor_data.pop(0)
            button_data.pop(0)

        clear_output(wait=True)
        fig, (ax1, ax2, ax3, ax4, ax5) = plt.subplots(5, 1, figsize=(11, 10), sharex=True)

        ax1.plot(time_series, asap_data, color='crimson', marker='o')
        ax1.set_title("1. Sensor Kebocoran Gas/Asap (MQ2)")
        ax1.set_yticks([0, 1])
        ax1.set_yticklabels(["AMAN", "ADA_ASAP"])
        ax1.grid(True)

        ax2.plot(time_series, pir_data, color='royalblue', marker='s')
        ax2.set_title("2. Sensor Gerakan (PIR)")
        ax2.set_yticks([0, 1])
        ax2.set_yticklabels(["NORMAL", "GERAKAN"])
        ax2.grid(True)

        ax3.plot(time_series, buzzer_data, color='purple', marker='*')
        ax3.set_title("3. Status Buzzer (Alarm)")
        ax3.set_yticks([0, 1])
        ax3.set_yticklabels(["DIAM", "BUNYI"])
        ax3.grid(True)

        ax4.plot(time_series, motor_data, color='darkorange', marker='^')
        ax4.set_title("4. Status Stepper Motor (Regulator Winn Gas W-688 M)")
        ax4.set_yticks([0, 1])
        ax4.set_yticklabels(["DIAM", "BERPUTAR"])
        ax4.grid(True)

        ax5.plot(time_series, button_data, color='forestgreen', marker='d')
        ax5.set_title("5. Tombol Reset Manual")
        ax5.set_yticks([0, 1])
        ax5.set_yticklabels(["RELEASED", "PRESSED"])
        ax5.set_xlabel("Waktu (Detik)")
        ax5.grid(True)

        plt.tight_layout()
        plt.show()
        plt.close(fig)  # free the figure; a new one is drawn every second

        time.sleep(1)

except KeyboardInterrupt:
    client.loop_stop()
    print("Visualisasi dihentikan.")
